# Справочники КУАП и ГК → DRP

Читает файлы из той же папки, что Excel-отчёты, и заливает:

- `sbx_da.acq_kuap_inn` — ИНН КУАП
- `sbx_da.acq_gk_inn` — все ИНН из файла ГК + флаг `is_exclude_gk`

Файл ГК содержит **все** группы. В исключения и таблицу ГК дашборда идут только:

Сирота, Агропромкомплектация, Триалспорт, Лаваши, Холдинг Днепровский.

Инструкция: `HOW_TO_exclusions_gk.md`.

## Перед запуском
`/home/jovyan/documents/Equaring/Data/inn_list_kuap.txt`  
`/home/jovyan/documents/Equaring/Data/gk_list_inn.xlsx`


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

KUAP_CANDIDATES = [
    DATA_DIR / 'inn_list_kuap.txt',
    DATA_DIR / 'in_listkuap.txt',
    DATA_DIR / 'inn_list_kuap.csv',
]
GK_CANDIDATES = [
    DATA_DIR / 'gk_list_inn.xlsx',
    DATA_DIR / 'gk_listinn.xlsx',
    DATA_DIR / 'gk_list_inn.xls',
]

drp_schema = 'sbx_da'
kuap_table = 'acq_kuap_inn'
gk_table = 'acq_gk_inn'
drp_superset_grant_role = 'raisa_superset'

# Только эти ГК идут в exclude_flag и в таблицу ГК на дашборде.
# needles — куски имени после нормализации (нижний регистр, без пробелов/ё).
EXCLUDE_GK_CANONICAL = [
    {'canonical': 'Сирота', 'needles': ['сирота']},
    {'canonical': 'Агропромкомплектация', 'needles': ['агропромкомплектац']},
    {'canonical': 'Триалспорт', 'needles': ['триалспорт']},
    {'canonical': 'Лаваши', 'needles': ['лаваши', 'лаваш']},
    {'canonical': 'Холдинг Днепровский', 'needles': ['днепровск', 'холдингднепров']},
]

print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
if DATA_DIR.exists():
    print('files in DATA_DIR:')
    for p in sorted(DATA_DIR.iterdir()):
        name = p.name.lower()
        if any(k in name for k in ('kuap', 'gk', 'инн', 'inn')):
            print(' ', p.name, p.stat().st_size, 'bytes')


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


def pick_col(columns, needles):
    lowered = {str(c): str(c).lower().replace('\n', ' ').strip() for c in columns}
    for col, low in lowered.items():
        if any(n in low for n in needles):
            return col
    return None


def norm_gk_name(v):
    if pd.isna(v):
        return ''
    s = str(v).lower().replace('ё', 'е')
    return re.sub(r'[^a-zа-я0-9]+', '', s)


def match_exclude_gk(name):
    n = norm_gk_name(name)
    if not n:
        return None
    for item in EXCLUDE_GK_CANONICAL:
        keys = [norm_gk_name(item['canonical'])] + [norm_gk_name(x) for x in item['needles']]
        keys = [k for k in keys if k]
        if any(k in n or n in k for k in keys):
            return item['canonical']
    return None


kuap_path = first_existing(KUAP_CANDIDATES)
gk_path = first_existing(GK_CANDIDATES)
print('KUAP file:', kuap_path)
print('GK file:  ', gk_path)
if kuap_path is None or gk_path is None:
    raise FileNotFoundError(
        'Не найден inn_list_kuap.txt или gk_list_inn.xlsx в DATA_DIR. '
        f'DATA_DIR={DATA_DIR}'
    )


In [ ]:
# КУАП: один ИНН на строку; допускаются csv / ; / tab
raw_kuap = pd.read_csv(kuap_path, header=None, dtype=object, sep=None, engine='python')
print('KUAP raw shape', raw_kuap.shape)
display(raw_kuap.head(8))

kuap_vals = []
for row in raw_kuap.itertuples(index=False):
    for cell in row:
        inn = normalize_inn_q1(cell)
        if inn:
            kuap_vals.append(inn)

# если первая строка была заголовком «ИНН» — она не пройдёт normalize
kuap_df = pd.DataFrame({'inn': kuap_vals}).drop_duplicates().sort_values('inn').reset_index(drop=True)
print('KUAP unique INN =', len(kuap_df))
if kuap_df.empty:
    raise RuntimeError('Список КУАП пуст после нормализации ИНН')
display(kuap_df.head(10))


In [ ]:
# ГК: авто-колонки ИНН + название группы
gk_raw = pd.read_excel(gk_path, dtype=object)
print('GK raw shape', gk_raw.shape)
print('GK columns:', list(gk_raw.columns))
display(gk_raw.head(8))

inn_col = pick_col(gk_raw.columns, ['инн', 'inn'])
name_col = pick_col(
    gk_raw.columns,
    ['гк', 'групп', 'group', 'gk_name', 'наимен', 'название', 'name', 'клиент'],
)
if inn_col is None:
    # fallback: колонка, где больше всего валидных ИНН
    best, best_n = None, -1
    for c in gk_raw.columns:
        n = gk_raw[c].map(normalize_inn_q1).notna().sum()
        if n > best_n:
            best, best_n = c, n
    if best_n <= 0:
        raise RuntimeError('В gk_list_inn.xlsx не найдена колонка ИНН')
    inn_col = best
if name_col is None or name_col == inn_col:
    name_candidates = [c for c in gk_raw.columns if c != inn_col]
    if not name_candidates:
        raise RuntimeError('В файле ГК нет колонки с названием группы')
    name_col = name_candidates[0]

print('resolved inn_col =', inn_col)
print('resolved name_col =', name_col)

gk_df = pd.DataFrame({
    'inn': gk_raw[inn_col].map(normalize_inn_q1),
    'gk_name_src': gk_raw[name_col].map(lambda x: None if pd.isna(x) else str(x).strip() or None),
})
gk_df = gk_df.dropna(subset=['inn', 'gk_name_src']).copy()
gk_df['gk_canonical'] = gk_df['gk_name_src'].map(match_exclude_gk)
gk_df['is_exclude_gk'] = gk_df['gk_canonical'].notna().map(lambda x: '1' if x else '0')
# для чарта ГК — каноническое имя; остальные группы оставляем как в файле
gk_df['gk_name'] = gk_df['gk_canonical'].fillna(gk_df['gk_name_src'])

dup_inn = gk_df['inn'].duplicated(keep=False)
if dup_inn.any():
    print('WARNING: один ИНН в нескольких ГК — приоритет у исключаемой ГК, иначе первое имя')
    display(gk_df.loc[dup_inn].sort_values(['inn', 'is_exclude_gk'], ascending=[True, False]))
    gk_df = gk_df.sort_values(['inn', 'is_exclude_gk'], ascending=[True, False])
    gk_df = gk_df.drop_duplicates(subset=['inn'], keep='first')

gk_df = gk_df[['inn', 'gk_name', 'gk_name_src', 'is_exclude_gk']].sort_values(
    ['is_exclude_gk', 'gk_name', 'inn'], ascending=[False, True, True]
).reset_index(drop=True)

print('GK unique INN =', len(gk_df), '| all groups =', gk_df['gk_name'].nunique())
print('exclude GK INN =', int((gk_df['is_exclude_gk'] == '1').sum()))

# сверка: каждая каноническая ГК должна найтись в файле
match_report = []
src_names = sorted(gk_df.loc[gk_df['is_exclude_gk'] == '1', 'gk_name_src'].dropna().unique())
for item in EXCLUDE_GK_CANONICAL:
    hit = gk_df[gk_df['gk_name'] == item['canonical']]
    match_report.append({
        'canonical': item['canonical'],
        'inns': int(len(hit)),
        'names_in_file': ', '.join(sorted(hit['gk_name_src'].unique())) if len(hit) else '',
    })
match_rep_df = pd.DataFrame(match_report)
display(match_rep_df)
missing = match_rep_df.loc[match_rep_df['inns'] == 0, 'canonical'].tolist()
if missing:
    print('WARNING: в файле не найдены ГК для исключений:', missing)
    print('Уникальные имена в файле (первые 80):')
    display(gk_df['gk_name_src'].drop_duplicates().sort_values().head(80).to_frame())

print('Имена из файла, попавшие в исключения:')
print(src_names)
display(gk_df.loc[gk_df['is_exclude_gk'] == '1'].groupby('gk_name', as_index=False).size())
display(gk_df.head(15))


In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')

drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
print('DRP connected as', drp_user)


In [ ]:
def upload_text_table(fq, df, grant=True):
    upload_df = df.copy()
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)
    col_defs = [f'"{str(c)}" TEXT' for c in upload_df.columns]
    create_sql = f'CREATE TABLE {fq} (\n  ' + ',\n  '.join(col_defs) + '\n)'
    with drp:
        drp.execute(f'DROP TABLE IF EXISTS {fq}')
        drp.execute(create_sql)
        drp.write(table=fq, df=upload_df, mode='append')
        cnt = drp.fetch(f'SELECT COUNT(*) AS row_cnt FROM {fq}')
        if grant:
            try:
                drp.execute(f'GRANT USAGE ON SCHEMA {drp_schema} TO {drp_superset_grant_role}')
                drp.execute(f'GRANT SELECT ON TABLE {fq} TO {drp_superset_grant_role}')
                print(f'OK: GRANT SELECT ON {fq} TO {drp_superset_grant_role}')
            except Exception as grant_exc:
                print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
                print(f'  GRANT SELECT ON TABLE {fq} TO {drp_superset_grant_role};')
    n = int(pd.to_numeric(cnt.iloc[0, 0], errors='coerce'))
    print(f'OK {fq} rows =', n)
    return n


kuap_fq = f'{drp_schema}.{kuap_table}'
gk_fq = f'{drp_schema}.{gk_table}'
upload_text_table(kuap_fq, kuap_df)
upload_text_table(gk_fq, gk_df[['inn', 'gk_name', 'is_exclude_gk']])


In [ ]:
# Smoke: КУАП и только исключаемые ГК (не весь файл)
smoke_sql = f'''
SELECT
  (SELECT COUNT(*) FROM {kuap_fq}) AS kuap_inns,
  (SELECT COUNT(*) FROM {gk_fq}) AS gk_inns_all,
  (SELECT COUNT(*) FROM {gk_fq} WHERE BTRIM(is_exclude_gk) = '1') AS gk_inns_exclude,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.inn AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {kuap_fq} k
      ON k.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
  ) AS datamart_kuap_inns,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.agr_id AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {kuap_fq} k
      ON k.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
  ) AS datamart_kuap_agr,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.inn AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {gk_fq} g
      ON g.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
     AND BTRIM(CAST(g.is_exclude_gk AS TEXT)) = '1'
  ) AS datamart_gk_excl_inns,
  (
    SELECT COUNT(DISTINCT NULLIF(BTRIM(CAST(d.agr_id AS TEXT)), ''))
    FROM sbx_da.tmp_shestopalov_acq_datamart_final_script_2 d
    JOIN {gk_fq} g
      ON g.inn = NULLIF(BTRIM(CAST(d.inn AS TEXT)), '')
     AND BTRIM(CAST(g.is_exclude_gk AS TEXT)) = '1'
  ) AS datamart_gk_excl_agr
'''
with drp:
    smoke = drp.fetch(smoke_sql)
    gk_preview = drp.fetch(
        f'''
        SELECT gk_name, is_exclude_gk, COUNT(*) AS inns
        FROM {gk_fq}
        WHERE BTRIM(is_exclude_gk) = '1'
        GROUP BY gk_name, is_exclude_gk
        ORDER BY inns DESC
        '''
    )
display(smoke)
display(gk_preview)

print('Next:')
print('1. SQL Lab: sources/sql/vd_acq_efficiency_flags.sql → dataset vd_acq_efficiency_flags')
print('2. Save as v2_filial_tsp_efficiency_excl (exclude_flag = 0)')
print('3. Save as v2_gk_tsp_efficiency (is_exclude_gk = 1, dimension gk_name)')
print('4. HOW_TO_exclusions_gk.md')
